# 06 — Resume mT5 fine-tuning (epochs 6–8)

Continues `checkpoint-23735` (epoch 5 of 8) to completion, plots the **full epoch 1–8 curve**,
then evaluates on the complete held-out test set under all four scoring protocols.

**Fixes applied vs. the previous version of this notebook:**

| severity | issue | fix |
|---|---|---|
| blocker | `test_df_sub` used one cell before it was defined | removed; evaluation uses the full test set |
| blocker | final eval ran on `test_df[:10]` — 10 examples | full 1,000 |
| major | `compute_rouge` defined twice, both naive `.split()` | one definition, four protocols |
| major | leftover debug cell overwrote `model` before Step 3 | removed |
| major | "optional eval-only" cell created a second, conflicting path | removed — this notebook only trains |
| moderate | plot mixed a step-axis and an epoch-index axis via `twiny().twinx()` | both series plotted against the real `epoch` field |
| moderate | nothing verified the checkpoint actually restored 5 epochs | pre-flight check in Step 5 |

If you want to evaluate a checkpoint *without* training, use `05_evaluate_checkpoint.ipynb`
instead — that is what it is for.

## Step 0 — Setup

In [ ]:
!nvidia-smi


`nvidia-smi` queries the driver directly, so it is ground truth for how many GPUs this session
has and whether anything is already holding memory on GPU 0. If GPU 0 already shows several GiB
in use, restart the kernel before continuing.

In [ ]:
!pip install -q evaluate rouge_score bert_score absl-py nltk qalsadi


In [ ]:
import os

print(f"CUDA_VISIBLE_DEVICES before override: {os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>')}")
# Kaggle sometimes provisions "GPU T4 x2"; Trainer would then fall back to legacy DataParallel
# and concentrate gradient buffers on GPU 0. Unconditional assignment (not setdefault) and
# before `import torch`, since CUDA_VISIBLE_DEVICES is read at CUDA init.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"CUDA_VISIBLE_DEVICES after override:  {os.environ['CUDA_VISIBLE_DEVICES']}")

import re
import json
import time
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import evaluate
import nltk
from nltk.stem import SnowballStemmer
from rouge_score import rouge_scorer
import qalsadi.lemmatizer
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Trainer, TrainingArguments, TrainerCallback,
    EarlyStoppingCallback, DataCollatorForSeq2Seq,
)
from transformers.optimization import Adafactor
from tqdm import tqdm

nltk.download("stopwords", quiet=True)
tqdm.pandas()
print(f"Visible GPUs: {torch.cuda.device_count()} (pinned to 1 by design)")

# ---------------- Config ----------------
MODEL_KEY = "mt5-small"
MODEL_NAME = "google/mt5-small"
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 100
GEN_MAX_LEN = 150
NUM_BEAMS = 4
BATCH_SIZE_GEN = 8

LEARNING_RATE = 5e-4      # T5-family convention; paired with Adafactor below, per the T5 recipe
NUM_EPOCHS = 8            # total across the whole run -- resume continues to this, not +8 more
TRAIN_BATCH_SIZE = 2      # fp32 (fp16 gives NaN on T5-family), so batch 2 x accum 8 = 16 effective
EVAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
RANDOM_SEED = 42

# TEST_MODE = True skips training entirely, loads the checkpoint directly, and evaluates on a
# small subset. Use it to validate this notebook end-to-end in ~2 minutes without a 7-hour run.
# Set back to False for the real run. Never interrupt training and then run the evaluation
# cells -- an interrupted trainer.train() leaves the in-memory model corrupted (it emits
# non-Arabic token spam), while the checkpoint on disk stays perfectly fine.
TEST_MODE = False
TEST_MODE_N = 20

DATA_DIR = "/kaggle/input/datasets/mohamedbentalb/arabic-summ-canonical-v1/canonical_dataset"
RESUME_FROM_CHECKPOINT = "/kaggle/input/models/mohamedbentalb/checkpoint-23735/transformers/default/1/results/checkpoints/checkpoint-23735"
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Fail fast with a directory listing rather than a cryptic error several cells later.
for label, path, marker in [("data", DATA_DIR, "manifest.json"),
                            ("checkpoint", RESUME_FROM_CHECKPOINT, "trainer_state.json")]:
    if path is None:
        continue
    if os.path.exists(os.path.join(path, marker)):
        print(f"{label}: OK")
    else:
        print(f"\nWARNING: no {marker} found for {label} at:\n  {path}\nSearching /kaggle ...")
        for base in ["/kaggle/input", "/kaggle/working"]:
            for dirpath, _, filenames in os.walk(base):
                if marker in filenames:
                    print(f"  Found: {dirpath}")
        print(f"Update the path above, then re-run this cell.")

print(f"\nModel: {MODEL_NAME} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")


## Step 1 — Load the canonical split

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

with open(f"{DATA_DIR}/manifest.json") as f:
    data_manifest = json.load(f)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"Canonical dataset built at: {data_manifest['created_at_utc']}")

assert set(train_df["article_hash"]).isdisjoint(set(test_df["article_hash"])), "train/test overlap!"
assert set(val_df["article_hash"]).isdisjoint(set(test_df["article_hash"])), "val/test overlap!"
print("Leakage re-check passed.")


## Step 2 — GPU memory check

If a previous cell crashed, its tensors stay referenced by the traceback and
`torch.cuda.empty_cache()` alone cannot reclaim them. This reports what is actually free.

In [ ]:
import gc, sys

for _v in ["model", "trainer", "optimizer"]:
    if _v in dir():
        exec(f"del {_v}")
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU 0: {alloc:.2f} GiB allocated of {total:.2f} GiB")
    print("Looks clean." if alloc < 1.0 else
          "WARNING: memory already in use -- restart the kernel before continuing.")


## Step 3 — Load model and tokenizer

**`model.tie_weights()` is deliberately NOT called here.** `checkpoint-23735`'s optimizer state
was saved against this model's untied parameter structure. Tying before the optimizer is built
changes how many distinct parameter tensors Adafactor sees, and `resume_from_checkpoint` then
fails with a parameter-group-size mismatch. Step 8 ties weights *after* training, so the saved
model is still correctly deduplicated (~1.1 GB rather than ~2.2 GB).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Loaded {MODEL_NAME} on {device}")
print("Note: Trainer will overwrite these weights from the checkpoint during resume.")


## Step 4 — Tokenize

Labels are **not** padded here. Padding is deferred to `DataCollatorForSeq2Seq`, which fills
padded label positions with `-100` so the loss ignores them. Padding labels to a fixed length
with the real pad-token id instead trains the model to predict "pad" everywhere and collapses
it to empty output.

In [ ]:
def tokenize_function(examples):
    model_inputs = tokenizer(examples["article"], max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(examples["summary"], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = Dataset.from_pandas(train_df[["article", "summary"]]).map(tokenize_function, batched=True)
eval_dataset  = Dataset.from_pandas(val_df[["article", "summary"]]).map(tokenize_function, batched=True)
print(f"train_dataset: {len(train_dataset)}  eval_dataset: {len(eval_dataset)}")


## Step 5 — Pre-flight: what is actually in the checkpoint

Reads `trainer_state.json` before training starts, so you know exactly how many epochs are
already done and how many this run will add. If this says 5, the run below trains 6, 7, 8.

In [ ]:
with open(f"{RESUME_FROM_CHECKPOINT}/trainer_state.json") as f:
    ckpt_state = json.load(f)

epochs_done = ckpt_state.get("epoch", 0)
steps_done = ckpt_state.get("global_step", 0)
prior_evals = [h["eval_loss"] for h in ckpt_state.get("log_history", []) if "eval_loss" in h]

print(f"Checkpoint: epoch {epochs_done:.2f}, global_step {steps_done}")
print(f"Validation losses already recorded: {len(prior_evals)}")
for i, v in enumerate(prior_evals, 1):
    print(f"  Epoch {i}: {v:.4f}")
print(f"\nThis run will train up to NUM_EPOCHS={NUM_EPOCHS}, i.e. ~{NUM_EPOCHS - epochs_done:.0f} more epochs.")

assert epochs_done < NUM_EPOCHS, (
    f"Checkpoint is already at epoch {epochs_done} and NUM_EPOCHS={NUM_EPOCHS}; "
    "nothing to train. Raise NUM_EPOCHS or use 05_evaluate_checkpoint.ipynb."
)


## Step 6 — Resume training

`fp16=False` because T5-family models produce NaN loss under fp16 — a documented, unfixed
architectural issue. bf16 is not an alternative here: it needs compute capability ≥ 8.0 and a
T4 is 7.5. Adafactor is constructed explicitly rather than via `optim="adafactor"`, because
Adafactor's own defaults (`relative_step=True`) compute an internal learning rate and would
silently ignore `LEARNING_RATE`.

Once this starts, let it run. Interrupting mid-step leaves the in-memory model in an
inconsistent state (files on disk are unaffected, but the loaded model becomes unusable until
reloaded).

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = TrainingArguments(
    output_dir=f"{RESULTS_DIR}/checkpoints",
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=1,
    seed=RANDOM_SEED,
)

optimizer = Adafactor(
    model.parameters(),
    lr=LEARNING_RATE,
    scale_parameter=False,
    relative_step=False,
    warmup_init=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

torch.cuda.empty_cache()
train_result = None
TRAINING_COMPLETED = False

if TEST_MODE:
    print("TEST_MODE -- skipping training. Loading checkpoint weights directly for evaluation.")
    model = AutoModelForSeq2SeqLM.from_pretrained(RESUME_FROM_CHECKPOINT).to(device)
    trainer.model = model
else:
    print(f"Resuming from {RESUME_FROM_CHECKPOINT}")
    print("Do NOT interrupt this cell. An interrupted run corrupts the in-memory model;")
    print("the evaluation cells below would then score garbage. Use TEST_MODE instead.")
    train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
    TRAINING_COMPLETED = True
    print(train_result)


## Step 7 — Full epoch 1–8 training curve

Both series are plotted against the **`epoch` field** from `trainer.state.log_history`, not
against a step index or a positional counter. That matters here: `resume_from_checkpoint`
restores the log history from epochs 1–5, which may have been logged at a different frequency
than epochs 6–8. Using the real epoch value puts every point on one honest axis regardless.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
if TEST_MODE and not history:
    # TEST_MODE never ran training, so pull the curve straight from the checkpoint's own state.
    history = ckpt_state.get("log_history", [])
    print("TEST_MODE: plotting the checkpoint's stored history (epochs 1-5).")
train_pts = [(h.get("epoch"), h["loss"]) for h in history if "loss" in h and "eval_loss" not in h and h.get("epoch") is not None]
eval_pts  = [(h.get("epoch"), h["eval_loss"]) for h in history if "eval_loss" in h and h.get("epoch") is not None]

print(f"log_history: {len(train_pts)} training points, {len(eval_pts)} validation points")
if eval_pts:
    print(f"epoch range: {min(e for e, _ in eval_pts):.1f} -> {max(e for e, _ in eval_pts):.1f}")

fig, ax1 = plt.subplots(figsize=(10, 5))
if train_pts:
    ax1.plot([e for e, _ in train_pts], [l for _, l in train_pts],
             color="tab:blue", alpha=0.55, linewidth=1, label="Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training loss", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()   # shares the SAME x-axis -- only the y-scale differs
if eval_pts:
    ax2.plot([e for e, _ in eval_pts], [l for _, l in eval_pts],
             color="tab:orange", marker="o", linewidth=2, label="Validation loss")
ax2.set_ylabel("Validation loss", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

# Mark where this session picked up from the checkpoint.
ax1.axvline(epochs_done, color="grey", linestyle="--", alpha=0.7)
ax1.text(epochs_done, ax1.get_ylim()[1], "  resume", va="top", fontsize=9, color="grey")

lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc="upper right")
plt.title(f"{MODEL_KEY} — training curve, epochs 1–{len(eval_pts)}")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/{MODEL_KEY}_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPer-epoch validation loss (full history, epochs 1-N):")
prev = None
for i, (ep, vl) in enumerate(eval_pts, 1):
    delta = f"{vl - prev:+.4f}" if prev is not None else "    --"
    print(f"  Epoch {i} (epoch={ep:.2f}): {vl:.4f}   {delta}")
    prev = vl


## Step 8 — Save the fine-tuned model

**`tie_weights()` must NOT be called here.** An earlier version of this notebook did, to
shrink the checkpoint from ~2.2 GB to ~1.1 GB, and it silently destroyed the model.

mT5 and AraT5 store `lm_head.weight` separately from `shared.weight`, and the two **diverge
during training** -- the output projection learns its own values. Tying performs
`setattr(parent, name, source_param)`, i.e. it *overwrites* `lm_head.weight` with
`shared.weight`, discarding everything the output head learned. Training loss stays perfect
and the model then emits non-Arabic token spam.

`transformers` guards against this, but only inside `from_pretrained`: it compares the
tensors, sees they differ, and refuses to tie -- that is the recurring "both are present in
the checkpoints ... we will NOT tie them" warning. A manual call passes `missing_keys=None`,
skipping that check and tying unconditionally.

The larger file size is therefore correct: these models genuinely need both tensors.

In [ ]:
if TEST_MODE:
    print("TEST_MODE -- skipping model save (nothing was trained).")
    model_save_path = None
else:
    model_save_path = f"{RESULTS_DIR}/{MODEL_KEY}_finetuned"
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    size_gb = sum(os.path.getsize(os.path.join(model_save_path, f))
                  for f in os.listdir(model_save_path)) / 1024**3
    print(f"Saved to {model_save_path} ({size_gb:.2f} GB)")


## Step 9 — Scoring protocols

One definition, four protocols. `rouge_score`'s stock tokenizer uses `NON_ALPHANUM_RE =
[^a-z0-9]+`, which reduces Arabic to an empty token list and returns exactly 0.0; its
`tokenizer=` argument also needs an object with `.tokenize()`, not a lambda. Both failures are
silent, so the self-test below asserts each protocol is non-zero on known-matching text.

In [ ]:
_PUNCT_RE = re.compile(r"([^\w\s])", flags=re.UNICODE)


def _basic_tokenize(text):
    """Lowercase, whitespace-split, split punctuation off as separate tokens (XL-Sum's BasicTokenizer)."""
    text = text.lower()
    text = _PUNCT_RE.sub(r" \1 ", text)
    return [t for t in text.split() if t]


class SplitTokenizer:
    name = "whitespace"
    def tokenize(self, text):
        return text.split()


class PunctTokenizer:
    name = "punct-aware"
    def tokenize(self, text):
        return _basic_tokenize(text)


class StemmedTokenizer:
    """Matches Kahla et al. (2023) via XL-Sum's multilingual_rouge_scoring."""
    name = "punct-aware + stemmed"
    def __init__(self):
        self.stemmer = SnowballStemmer("arabic", ignore_stopwords=True)
    def tokenize(self, text):
        return [self.stemmer.stem(t) for t in _basic_tokenize(text)]


class LemmaTokenizer:
    """Lemmatisation resolves to real dictionary entries; the stemmer truncates
    (باريس -> اريس). ~180x slower per call, so results are memoised."""
    name = "punct-aware + lemmatized"
    def __init__(self):
        self._lem = qalsadi.lemmatizer.Lemmatizer()
        self._cache = {}
    def _lemma(self, w):
        if w not in self._cache:
            try:
                self._cache[w] = self._lem.lemmatize(w) or w
            except Exception:
                self._cache[w] = w
        return self._cache[w]
    def tokenize(self, text):
        return [self._lemma(t) for t in _basic_tokenize(text)]


PROTOCOLS = [SplitTokenizer(), PunctTokenizer(), StemmedTokenizer(), LemmaTokenizer()]
ROUGE_KEYS = ["rouge1", "rouge2", "rougeL", "rougeLsum"]


def rouge_with(predictions, references, tokenizer_obj):
    scorer = rouge_scorer.RougeScorer(ROUGE_KEYS, use_stemmer=False, tokenizer=tokenizer_obj)
    totals = {k: [] for k in ROUGE_KEYS}
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        for k in ROUGE_KEYS:
            totals[k].append(s[k].fmeasure)
    return {k: float(np.mean(v)) for k, v in totals.items()}


def compute_bertscore(predictions, references, lang="ar"):
    bs = evaluate.load("bertscore")
    r = bs.compute(predictions=predictions, references=references, lang=lang)
    return r  # per-example lists, so slices can be averaged without recomputing


_p = "أعلن متحدث باسم فرق الإطفاء أنّ الحريق الضخم اندلع في كاتدرائية نوتردام"
_r = "أعلنت فرق الإطفاء أنّ الحريق الضخم الذي اندلع في كاتدرائية نوتردام في باريس"
print("Self-test (all must be > 0):")
for _t in PROTOCOLS:
    _v = rouge_with([_p], [_r], _t)["rouge1"]
    print(f"  {_t.name:<26} rouge1 = {_v:.4f}")
    assert _v > 0, f"{_t.name} returned zero on matching Arabic -- tokenizer broken"


## Step 10 — Final evaluation on the **full** test set

The previous version of this notebook evaluated on `test_df[:10]`. This uses all 1,000.

In [ ]:
test_articles = test_df["article"].tolist()
test_references = test_df["summary"].tolist()
source_vals = test_df["source"].tolist() if "source" in test_df.columns else ["all"] * len(test_df)
assert len(test_articles) == len(test_df), "evaluating on a subset -- use the full test set"


def generate_summaries(model, tokenizer, articles, batch_size=BATCH_SIZE_GEN):
    model.eval()
    preds = []
    for i in tqdm(range(0, len(articles), batch_size), desc="Generating"):
        batch = articles[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True,
                           padding=True, max_length=MAX_INPUT_LEN).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_length=GEN_MAX_LEN,
                                 num_beams=NUM_BEAMS, early_stopping=True)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds


def arabic_ratio(text):
    """Fraction of characters in the Arabic Unicode block (U+0600-U+06FF)."""
    if not text or not text.strip():
        return 0.0
    return sum(1 for ch in text if "؀" <= ch <= "ۿ") / len(text)


# --- Sanity gate: 3 generations (~5s) before committing to 1000 (~8 min). ---
# An interrupted trainer.train() leaves the model emitting non-Arabic token spam
# ("ILLARILLARILLAR..."). Catching that here costs seconds instead of minutes.
_probe = generate_summaries(model, tokenizer, test_articles[:3])
_ratios = [arabic_ratio(p) for p in _probe]
print("Sanity probe -- Arabic character ratio per sample:")
for _i, (_p, _r) in enumerate(zip(_probe, _ratios)):
    print(f"  [{_i}] ratio={_r:.2f}  {_p[:90]}")

assert np.mean(_ratios) > 0.5, (
    "Model output is not Arabic (mean ratio "
    f"{np.mean(_ratios):.2f}). This is what a corrupted in-memory model looks like -- almost "
    "always because trainer.train() was interrupted. The checkpoint on disk is fine: restart "
    "the kernel and either re-run training uninterrupted, or set TEST_MODE = True."
)
print(f"Sanity probe passed (mean Arabic ratio {np.mean(_ratios):.2f}).")

if TEST_MODE:
    test_articles = test_articles[:TEST_MODE_N]
    test_references = test_references[:TEST_MODE_N]
    source_vals = source_vals[:TEST_MODE_N]
    print(f"TEST_MODE: evaluating on {len(test_articles)} examples only.")

t0 = time.time()
predictions = generate_summaries(model, tokenizer, test_articles)
generation_time_s = time.time() - t0

empty = sum(1 for p in predictions if not p.strip())
print(f"\nGenerated {len(predictions)} summaries in {generation_time_s:.0f}s | empty: {empty}")

print("\n=== Sample predictions (read these before trusting any metric) ===")
for i in range(3):
    print(f"\n[{i}] PRED: {predictions[i][:200]}")
    print(f"[{i}] REF : {test_references[i][:200]}")


In [ ]:
slices = {"ALL": list(range(len(predictions)))}
for src in sorted(set(source_vals)):
    slices[src] = [i for i, s in enumerate(source_vals) if s == src]

rows = []
for slice_name, idx in slices.items():
    sp = [predictions[i] for i in idx]
    sr = [test_references[i] for i in idx]
    for tok in PROTOCOLS:
        r = rouge_with(sp, sr, tok)
        rows.append({"slice": slice_name, "n": len(idx), "protocol": tok.name,
                     "ROUGE-1": r["rouge1"] * 100, "ROUGE-2": r["rouge2"] * 100,
                     "ROUGE-L": r["rougeL"] * 100})

results_df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.2f}")
print(results_df.to_string(index=False))


In [ ]:
METRIC_COLS = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
all_rows = results_df[results_df["slice"] == "ALL"].set_index("protocol")[METRIC_COLS]

print(f"{'protocol':<28}{'ROUGE-1':>10}{'ROUGE-2':>10}{'ROUGE-L':>10}")
for t in PROTOCOLS:
    r = all_rows.loc[t.name]
    print(f"{t.name:<28}{r['ROUGE-1']:>10.2f}{r['ROUGE-2']:>10.2f}{r['ROUGE-L']:>10.2f}")
print("-" * 58)
total = all_rows.loc["punct-aware + lemmatized"] - all_rows.loc["whitespace"]
lemma_vs_stem = all_rows.loc["punct-aware + lemmatized"] - all_rows.loc["punct-aware + stemmed"]
print(f"{'lemmatized - stemmed':<28}{lemma_vs_stem['ROUGE-1']:>+10.2f}{lemma_vs_stem['ROUGE-2']:>+10.2f}{lemma_vs_stem['ROUGE-L']:>+10.2f}")
print(f"{'TOTAL swing':<28}{total['ROUGE-1']:>+10.2f}{total['ROUGE-2']:>+10.2f}{total['ROUGE-L']:>+10.2f}")
rel = [((all_rows.loc["punct-aware + lemmatized"][m] / all_rows.loc["whitespace"][m] - 1) * 100
        if all_rows.loc["whitespace"][m] > 0 else float("nan")) for m in METRIC_COLS]
print(f"{'TOTAL relative':<28}{rel[0]:>+9.1f}%{rel[1]:>+9.1f}%{rel[2]:>+9.1f}%")


In [ ]:
bs_raw = compute_bertscore(predictions, test_references)
bert_rows = [{"slice": s, "n": len(idx),
              "precision": float(np.mean([bs_raw["precision"][i] for i in idx])),
              "recall": float(np.mean([bs_raw["recall"][i] for i in idx])),
              "f1": float(np.mean([bs_raw["f1"][i] for i in idx]))}
             for s, idx in slices.items()]
bert_df = pd.DataFrame(bert_rows)
print(bert_df.to_string(index=False))


## Step 11 — Save results

In [ ]:
# TEST_MODE writes to a separate prefix so a smoke run can never overwrite real results.
SUFFIX = "_TESTMODE" if TEST_MODE else ""

pd.DataFrame({
    "article": test_articles,
    "reference": test_references,
    "prediction": predictions,
    "source": source_vals,
}).to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_predictions.csv", index=False)

results_df.to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_rouge_by_protocol.csv", index=False)
bert_df.to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_bertscore.csv", index=False)

payload = {
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "run_completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data_manifest_created_at": data_manifest["created_at_utc"],
    "resumed_from": RESUME_FROM_CHECKPOINT,
    "epochs_in_checkpoint": epochs_done,
    "hyperparameters": {
        "learning_rate": LEARNING_RATE, "num_epochs": NUM_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
        "max_input_len": MAX_INPUT_LEN, "max_target_len": MAX_TARGET_LEN,
        "num_beams": NUM_BEAMS, "seed": RANDOM_SEED, "fp16": False,
        "optimizer": "Adafactor(scale_parameter=False, relative_step=False, warmup_init=False)",
    },
    "dataset_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "training": {
        "test_mode": TEST_MODE,
        "training_completed": TRAINING_COMPLETED,
        "final_train_loss": (train_result.training_loss if train_result is not None else None),
        "log_history": history,
        "validation_loss_per_epoch": [l for _, l in eval_pts],
    },
    "rouge_by_protocol": results_df.to_dict(orient="records"),
    "bertscore_by_slice": bert_df.to_dict(orient="records"),
    "generation": {"time_s": generation_time_s, "empty_outputs": int(empty)},
    "evaluation_notes": (
        "ROUGE via rouge_score with Unicode-safe tokenizers; the stock default empties Arabic. "
        "'stemmed' replicates Kahla et al. (2023) / XL-Sum multilingual_rouge_scoring. All "
        "protocols share one tokenization base, so normalisation is the only variable."
    ),
}

with open(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_results.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Saved to {RESULTS_DIR}/:")
for fn in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {fn}")


## Next

Commit this notebook so `results/` is preserved as an output, then upload
`mt5-small_finetuned` as a Kaggle Model if you want to reuse it later.

`mt5-small_predictions.csv` can be re-scored at any time without regenerating — useful if you
add a fifth protocol later.